# Our own Santali voice: fine-tune MMS-TTS (VITS) on one IndicVoices-R Santali speaker (C4)

* **Base:** `facebook/mms-tts-unr` (Mundari, a Munda language like Santali; CC BY-NC 4.0; character
  input in **Odia script**). There is no `mms-tts-sat`.
* **Text:** Ol Chiki → Devanagari (`translit/olchiki.py`) → Odia script (`translit/odia.py`), so the base
  vocabulary is kept (character-level, no espeak-ng; espeak-ng has no Santali).
* **Data:** `ai4bharat/indicvoices_r`, config **Santali** (CC BY 4.0, gated: accept the terms on the
  dataset page with your account first). One speaker: the one with the most clean hours
  (SNR ≥ 25 dB, 1–20 s clips). Speaker ID, hours and licence are written to `speaker_selection.json`.
* **Training:** `ylacombe/finetune-hf-vits` @ `6f3f51f` (MIT), with the discriminator converted from
  the original MMS checkpoint (`convert_original_discriminator_checkpoint.py --language_code unr`).
* **Export:** `tools/export/export_mms_vits_onnx.py` (this repo), tested on the laptop with mms-tts-unr.
* It ships only if it beats the live voice on the A6 round-trip CER, with a model card.

**Run on Kaggle (exact steps)**
1. kaggle.com → Settings → Phone verification done (needed for GPU and Internet).
2. Create → New Notebook → File → **Import Notebook** → upload this `.ipynb`.
3. Right panel → Session options → **Accelerator: GPU T4 x2** (or P100); **Internet: On**.
4. Add-ons → **Secrets** → Add secret: label **`HF_TOKEN`**, value = a Hugging Face **read** token → tick it
   for this notebook. (The code reads it with `UserSecretsClient`; it is never printed or saved.)
5. On huggingface.co/datasets/ai4bharat/indicvoices_r the terms must be accepted by the token's account
   (already done for this team on 25 Sep).
6. **Save Version → Save & Run All (Commit)**.
7. At the end download `santali_voice_onnx.zip` (model.onnx, tokens.txt, tts.json, tokenizer,
   speaker_selection.json, samples) into the repo's `dist/santali_voice/` and tell me.
8. If it stops (the 12 h limit): Add Input → the stopped version's output → Save & Run All again;
   finished shards are skipped and training resumes from the newest `checkpoint-*`.

**Expected runtime (T4):** metadata scan of 108 shards (columns only) ≈ 10–15 min · downloading the
chosen speaker's shards ≈ 20–60 min (depends on how many shards the speaker spans; the train split is
38.8 GB in total, only the needed shards are read) · training ≈ 2–3 h for 150 epochs on ~1–2 h of audio ·
export 5 min. **≈ 3–4.5 h.** Estimates, not measurements. A Kaggle session is limited to 12 h.

In [ ]:
REPO_URL = "https://github.com/barath0512s-rgb/sih-hackathon"

import os, sys, subprocess, json, glob, time, hashlib, random
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
import torch
assert torch.cuda.is_available(), "No GPU: Kaggle -> Settings -> Accelerator -> GPU; Colab -> Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/nijbhasha"
else:
    PERSIST = "/kaggle/working"            # saved as the notebook's output; re-attach it to resume
os.makedirs(PERSIST, exist_ok=True)
# Resume on Kaggle: an earlier run's output attached as an input is copied back into
# /kaggle/working, so finished steps are skipped and training continues from its checkpoints.
import shutil
for prev in glob.glob("/kaggle/input/*/"):
    if any(os.path.exists(os.path.join(prev, m)) for m in ("lora_hi_unr", "lora_unr_hi", "santali_voice_run", "sat_meta.json")):
        print("resuming from", prev)
        shutil.copytree(prev, PERSIST, dirs_exist_ok=True)
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "repo"], check=True)
sys.path.insert(0, os.path.abspath("repo"))

In [ ]:
!pip install -q transformers==4.46.1 datasets==3.1.0 accelerate==1.0.1 huggingface_hub==0.26.2 pyarrow==17.0.0 \
    soundfile==0.12.1 librosa==0.10.2.post1 Cython==3.0.11 onnx==1.17.0 onnxruntime==1.19.2 "numpy<2" matplotlib tensorboard
if not os.path.exists("finetune-hf-vits"):
    subprocess.run("git clone -q https://github.com/ylacombe/finetune-hf-vits && cd finetune-hf-vits && "
                   "git checkout -q 6f3f51f4d667f5c3eef89484d151ffd39d2c2b89 && "
                   "cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace",
                   shell=True, check=True)

In [ ]:
# Hugging Face token. On Kaggle: from Kaggle Secrets (Add-ons -> Secrets, label HF_TOKEN).
# Elsewhere: typed, hidden. Kept only in this process's memory: never printed, never written.
if not os.environ.get("HF_TOKEN"):
    if os.path.exists("/kaggle"):
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        import getpass
        os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face read token (input hidden): ").strip()
from huggingface_hub import whoami
print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])   # the account name only

In [ ]:
# 1. Speaker selection: read only the metadata columns of every Santali train shard.
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem
fs = HfFileSystem(token=os.environ["HF_TOKEN"])
shards = sorted(fs.glob("datasets/ai4bharat/indicvoices_r/Santali/train-*.parquet"))
print(len(shards), "shards")
meta_path = f"{PERSIST}/sat_meta.json"
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
for i, sh in enumerate(shards):
    if sh in meta: continue
    with fs.open(sh, "rb") as f:
        t = pq.read_table(f, columns=["speaker_id", "duration", "snr", "gender", "text"]).to_pandas()
    meta[sh] = t.assign(text_len=t.text.str.len()).drop(columns=["text"]).to_dict("list")
    if i % 10 == 0: json.dump(meta, open(meta_path, "w")); print(i, end=" ", flush=True)
json.dump(meta, open(meta_path, "w"))
import pandas as pd
rows = pd.concat([pd.DataFrame(v).assign(shard=k) for k, v in meta.items()])
ok = rows[(rows.snr >= 25) & (rows.duration.between(1, 20))]
by = ok.groupby("speaker_id").agg(hours=("duration", lambda d: d.sum() / 3600), clips=("duration", "size"),
                                  gender=("gender", "first"), shards=("shard", "nunique")).sort_values("hours", ascending=False)
print(by.head(10))
SPEAKER = by.index[0]             # override here to choose another speaker
sel = {"dataset": "ai4bharat/indicvoices_r", "config": "Santali", "split": "train", "licence": "CC BY 4.0 (attribution)",
       "speaker_id": str(SPEAKER), "hours": round(float(by.loc[SPEAKER, "hours"]), 2),
       "clips": int(by.loc[SPEAKER, "clips"]), "gender": str(by.loc[SPEAKER, "gender"]),
       "filters": "snr >= 25 dB, 1-20 s", "top10": by.head(10).reset_index().to_dict("records")}
json.dump(sel, open(f"{PERSIST}/speaker_selection.json", "w"), indent=1, default=str)
print(sel["speaker_id"], sel["hours"], "h")

In [ ]:
# 2. Extract that speaker's clips (16 kHz wav) + Ol Chiki -> Odia text. Resumable per shard.
import io, soundfile as sf, librosa, numpy as np
from translit.odia import olchiki_to_odia
DATA = f"{PERSIST}/sat_speaker"; os.makedirs(f"{DATA}/wav", exist_ok=True)
need = sorted(ok[ok.speaker_id == SPEAKER].shard.unique())
done_path = f"{DATA}/done_shards.json"
done = set(json.load(open(done_path))) if os.path.exists(done_path) else set()
lines_path = f"{DATA}/lines.jsonl"
for sh in need:
    if sh in done: continue
    with fs.open(sh, "rb") as f:
        t = pq.read_table(f, columns=["speaker_id", "duration", "snr", "text", "audio"]).to_pandas()
    t = t[(t.speaker_id == SPEAKER) & (t.snr >= 25) & t.duration.between(1, 20)]
    with open(lines_path, "a", encoding="utf-8") as out:
        for _, r in t.iterrows():
            a, sr = sf.read(io.BytesIO(r.audio["bytes"]), dtype="float32")
            if a.ndim > 1: a = a.mean(1)
            a = librosa.resample(a, orig_sr=sr, target_sr=16000)
            name = hashlib.sha1(r.audio["bytes"][:4096]).hexdigest()[:16] + ".wav"
            sf.write(f"{DATA}/wav/{name}", a, 16000, subtype="PCM_16")
            out.write(json.dumps({"file_name": f"wav/{name}", "olchiki": r.text,
                                  "text": olchiki_to_odia(r.text)}, ensure_ascii=False) + "\n")
    done.add(sh); json.dump(sorted(done), open(done_path, "w")); print("shard done", sh.rsplit("/", 1)[1])
lines = [json.loads(l) for l in open(lines_path, encoding="utf-8")]
lines = list({l["file_name"]: l for l in lines}.values())
print(len(lines), "clips")

In [ ]:
# 3. Vocabulary check against the base model, hold out 20 clips for listening/CER, write metadata.csv.
from transformers import AutoTokenizer
base_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-unr")
vocab = set(base_tok.get_vocab())
oov = {}
for l in lines:
    for c in l["text"]:
        if c != " " and c not in vocab: oov[c] = oov.get(c, 0) + 1
chars = sum(len(l["text"]) for l in lines)
print("out-of-vocabulary characters (dropped by the tokenizer):", oov, f"= {sum(oov.values()) / chars:.2%} of characters")
random.seed(20260926); random.shuffle(lines)
held, train = lines[:20], lines[20:]
import csv
with open(f"{DATA}/metadata.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["file_name", "text"])
    for l in train: w.writerow([l["file_name"], l["text"]])
json.dump(held, open(f"{PERSIST}/held_out_clips.json", "w"), ensure_ascii=False, indent=1)
print(len(train), "train clips,", len(held), "held out")

In [ ]:
# 4. Base checkpoint with a discriminator (converted from the original MMS unr checkpoint, CC BY-NC 4.0).
BASE_TRAIN = f"{PERSIST}/mms-unr-train"
if not os.path.exists(f"{BASE_TRAIN}/config.json"):
    subprocess.run(["python", "convert_original_discriminator_checkpoint.py", "--language_code", "unr",
                    "--pytorch_dump_folder_path", BASE_TRAIN], cwd="finetune-hf-vits", check=True)

In [ ]:
# 5. Fine-tune. Resumes from the newest checkpoint-* in OUT.
OUT = f"{PERSIST}/santali_voice_run"
cfg = {
  "project_name": "nijbhasha_santali_voice", "push_to_hub": False, "report_to": ["tensorboard"],
  "overwrite_output_dir": False, "output_dir": OUT, "resume_from_checkpoint": "latest",
  "dataset_name": DATA, "audio_column_name": "audio", "text_column_name": "text",
  "train_split_name": "train", "eval_split_name": "train",
  "full_generation_sample_text": held[0]["text"],
  "max_duration_in_seconds": 20, "min_duration_in_seconds": 1.0, "max_tokens_length": 500,
  "model_name_or_path": BASE_TRAIN, "preprocessing_num_workers": 2,
  "do_train": True, "num_train_epochs": 150, "gradient_accumulation_steps": 1, "gradient_checkpointing": False,
  "per_device_train_batch_size": 16, "learning_rate": 2e-5, "adam_beta1": 0.8, "adam_beta2": 0.99,
  "warmup_ratio": 0.01, "group_by_length": False,
  "do_eval": True, "eval_steps": 200, "per_device_eval_batch_size": 16, "max_eval_samples": 8,
  "do_step_schedule_per_epoch": True, "save_steps": 500, "save_total_limit": 2,
  "weight_disc": 3, "weight_fmaps": 1, "weight_gen": 1, "weight_kl": 1.5, "weight_duration": 1, "weight_mel": 35,
  "fp16": True, "seed": 456}
json.dump(cfg, open("finetune-hf-vits/santali.json", "w"), ensure_ascii=False, indent=1)
t0 = time.time()
subprocess.run("accelerate launch run_vits_finetuning.py santali.json", shell=True, cwd="finetune-hf-vits", check=True)
print(f"training {time.time() - t0:.0f} s")

In [ ]:
# 6. Export (the repo's exporter) + samples of the 20 held-out sentences, then zip.
EXP = f"{PERSIST}/santali_voice_onnx"
subprocess.run([sys.executable, "repo/tools/export/export_mms_vits_onnx.py", "--model", OUT, "--out", EXP,
                "--check-text", held[0]["text"]], check=True)
from transformers import VitsModel
m = VitsModel.from_pretrained(OUT).eval().cuda(); tk = AutoTokenizer.from_pretrained(OUT)
os.makedirs(f"{EXP}/samples", exist_ok=True)
for i, l in enumerate(held):
    with torch.no_grad():
        w = m(**tk(l["text"], return_tensors="pt").to("cuda")).waveform[0].cpu().numpy()
    sf.write(f"{EXP}/samples/{i:02d}.wav", w, m.config.sampling_rate)
json.dump(held, open(f"{EXP}/samples/texts.json", "w"), ensure_ascii=False, indent=1)
subprocess.run(f"cp {PERSIST}/speaker_selection.json {EXP}/ && cd {PERSIST} && zip -qr santali_voice_onnx.zip santali_voice_onnx", shell=True, check=True)
print(os.path.getsize(f"{PERSIST}/santali_voice_onnx.zip") / 1e6, "MB")